# Chapter 7: Search In-Depth
## Topic 5: Keyword Rules Reframed — Hand-Rolled Classifier vs BM25 Side by Side

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- This topic isn't a new algorithm — it's a precise, term-by-term comparison of a rule-based keyword classifier (built earlier in the project) against BM25, to answer a question every retrieval interview eventually asks: "why not just do keyword matching?"
- The rule-based classifier works like this: check if any word from a curated list of FD-related terms appears in the email, check if any word from a curated list of "negation" terms appears (terms that suggest the email is NOT about FDs, like "loan" or "login"), and combine those two yes/no signals into one of four outcomes: FD, Non-FD, Multiple Category, or ambiguous.
- Every earlier topic in this chapter has casually called this "the crudest possible form of sparse retrieval" — this topic makes that comparison precise and quantified instead of just asserting it.
- The real value: understanding exactly which parts of BM25's math the rule engine already approximates, which parts it skips entirely, and why skipping them was actually the right call for a binary classifier — even though skipping the same things would be the wrong call for ranked retrieval.


### 2. Internal Working — Term-by-Term Comparison

**Term presence — the rule engine has this, in a weaker form**

- The rule engine checks: does this exact word or phrase appear anywhere in the text? A simple yes/no.
- BM25 starts from that exact same binary fact, but only as the first input to a much larger formula — it doesn't stop there.

**Term frequency weighting — the rule engine has none**

- The rule engine's "any match" check stops as soon as it finds the first match. Whether a keyword appears once or eleven times, the yes/no flag is identical either way.
- BM25 explicitly weights how many times a term appears, with diminishing returns for repetition (the saturation curve from Topic 1).
- Real practical consequence: an email that mentions a negation term once, in passing, gets vetoed by the rule engine exactly as strongly as an email that's entirely about that negation term. The rule engine can't distinguish "mentioned once, briefly" from "this is what the whole email is about."

**Term rarity (inverse document frequency) — the rule engine has none**

- The rule engine treats every keyword in its list as equally diagnostic — a rare, highly specific term counts exactly the same as a more generic, commonly-used phrase.
- BM25 automatically gives more weight to rarer terms, recalculated fresh from actual corpus statistics every time the corpus changes.
- Real practical consequence: the rule engine's per-keyword importance was effectively hand-tuned once by a human looking at word-frequency counts during development. BM25 recomputes this signal automatically and continuously.

**Document length normalization — the rule engine doesn't need it much here**

- The rule engine operates on short customer emails of fairly similar length, so length differences matter far less here than they would for a knowledge base mixing short FAQ answers with long multi-step procedures.
- This is the one BM25 component that's genuinely less relevant to the rule engine's original use case (classifying short, similarly-sized emails), even though it matters a lot for retrieval over a knowledge base with widely varying document lengths.

**Negation handling — the rule engine has this, and BM25 has no equivalent**

- The rule engine's negation check is a second, independent signal that can actively override or combine with the main signal — this is genuine domain logic that plain BM25 simply doesn't model.
- BM25 has no built-in concept of "this term, if present, should suppress relevance" — a standard BM25 score just adds up positive term contributions; there's no subtraction mechanism.
- This is worth stating clearly in interviews: the rule engine isn't just "a weaker version of BM25" — it also has structured domain logic (routing to multiple distinct output categories based on combining two signals) that a raw BM25 score doesn't provide out of the box.


### 3. How This Is Implemented in This Project

- The rule engine's role doesn't change because of anything in this chapter — it remains a fast, free, zero-latency first-pass filter in the email classification pipeline, entirely separate from this chapter's BM25 work, which targets a different task: ranked retrieval of policy document chunks for the generation pipeline, not email classification.
- These are two separate systems that happen to share underlying ideas — this comparison is educational, not a suggestion to replace one with the other.
- Where a BM25-style upgrade of the rule engine actually would make sense: if the classifier's false-positive rate on genuinely ambiguous emails needed to be reduced without adding a full language-model call, replacing the simple yes/no flag with an actual BM25 relevance score compared against a tunable threshold is the natural next step.
- This would mean building two small BM25 indexes — one representing the FD-related keyword groups, one representing the negation terms — scoring each incoming email against both, and using the actual score magnitude instead of binary presence. This would catch exactly the "mentioned once in passing" vs "this is the whole point of the email" distinction that the current rule engine structurally cannot.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **False positives from single incidental mentions:** the rule engine's biggest structural weakness is treating "mentioned once, briefly" the same as "this is the actual topic" — this is a known, predictable source of misclassification, not a rare bug.
- **Rule maintenance cost:** every time new keyword variants emerge (new product names, new slang, new spelling variants), someone has to manually update the keyword and negation lists — this is an ongoing maintenance cost that a statistically-driven approach like BM25 doesn't have, since BM25 recalculates term importance automatically from the corpus.
- **Debugging a misclassification:** because the rule engine's logic is fully transparent (just word lists and boolean logic), debugging why a specific email was misclassified is usually fast — check which exact keyword or negation term matched, unlike debugging an opaque machine learning or LLM-based classifier.
- **Latency and cost:** the rule engine is about as fast and cheap as a classifier can be — no model to load, no API call, pure string matching. This is a real, measurable advantage over any BM25-scored or ML-based replacement, which is why replacing it should only happen if the accuracy gain is worth the added complexity and latency.
- **Scaling:** the rule engine's cost doesn't meaningfully increase with corpus size, since it doesn't depend on corpus statistics at all — it just checks a fixed, small list of terms against each incoming email. A BM25-based version would need its index rebuilt or updated as new labeled data comes in.
- **Monitoring:** track the "Multiple Category" and "ambiguous" classification rates over time — a rising rate of either suggests the keyword or negation lists need updating, or that a more nuanced approach (like the BM25-scored variant) is becoming necessary.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Simplicity and speed vs nuance:** the rule engine trades away frequency weighting, rarity weighting, and length normalization in exchange for being extremely fast, free, and fully transparent. For a first-pass filter on short, fairly similar-length emails, this trade-off is reasonable — length normalization in particular wouldn't have added much value here anyway.
- **Binary domain logic vs continuous scoring:** the rule engine's negation-based branching into multiple output categories is genuinely useful domain logic that a raw BM25 score doesn't provide natively. Any BM25-based replacement would need to preserve this branching structure on top of the new scores, not just report a single relevance number.
- **When the trade-off stops being worth it:** if the false-positive rate from single incidental mentions becomes a real, measured business problem, upgrading the binary flags to BM25-based scores (while keeping the same overall decision structure) is a low-cost way to add frequency and rarity awareness without discarding the domain logic that makes the rule engine effective in the first place.

### 6. Alternatives and When to Use Each

- **Pure binary keyword rules (the current approach):** best for a fast, transparent, zero-cost first-pass filter where the corpus is short and fairly uniform in length, and where occasional false positives from incidental mentions are an acceptable trade-off for speed and simplicity.
- **BM25-scored variant (same decision structure, frequency/rarity-aware scores):** the natural upgrade path when false positives from single incidental mentions become a measured problem, without wanting to introduce a full machine learning model or LLM call.
- **A full statistical or ML classifier:** worth considering only if the BM25-scored variant still isn't accurate enough, and if there's enough labeled data and infrastructure appetite to justify a heavier solution.


### 7. Common Mistakes and Production Failures

- Assuming the rule engine is "just a worse BM25" and therefore always worth replacing — this misses that it also encodes genuine domain logic (multi-way branching using two independent signals) that a raw BM25 score doesn't provide by itself.
- Letting the keyword and negation lists go stale as new terminology, slang, or product names emerge, without a process for periodically reviewing and updating them.
- Treating a single incidental mention of a negation term as equally strong evidence as an email entirely about that topic — this is the rule engine's known, predictable failure mode, and any production monitoring should specifically watch for this pattern.
- Replacing the rule engine with something more sophisticated without first confirming that the extra accuracy is actually worth the added latency, cost, and complexity — the rule engine's speed and transparency are real, valuable properties that shouldn't be discarded lightly.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why is a hand-rolled keyword classifier considered a form of sparse retrieval?
  A: Both work by checking word presence in a mostly-empty space of possible words — the classifier just stops at a binary yes/no per keyword, while BM25 continues on to weight by frequency and rarity. They share the same starting point but BM25 goes much further.

- Q: What's the single biggest structural weakness of a binary keyword classifier compared to BM25?
  A: It can't distinguish between a term mentioned once in passing and a term that's the entire subject of the text — frequency doesn't matter at all once a match is found, whereas BM25 explicitly weights how often a term appears.

**Intermediate**

- Q: If you wanted to add BM25-style scoring to an existing rule-based classifier without breaking its output categories, how would you do it?
  A: Build two small BM25 indexes — one over the positive-signal keyword groups, one over the negation terms — score the incoming text against each, and replace the original binary flags with score-above-threshold checks, keeping the same branching logic that produces the same set of output categories.

- Q: Does BM25 handle negation the way this rule engine does?
  A: No — BM25 only adds up positive term contributions; there's no built-in mechanism for one term to suppress or override another term's contribution. The rule engine's negation-based routing is genuine domain logic that would need to be layered on top of BM25 scores, not something BM25 provides natively.

**Advanced**

- Q: A teammate says "just replace the keyword rules with BM25, it's strictly better." How do you respond?
  A: BM25 is better at frequency and rarity weighting, but it doesn't replace the rule engine's negation-based branching logic on its own — that structured domain logic (routing to multiple categories based on combining two independent signals) would need to be preserved separately. Also, the rule engine's speed, zero training cost, and full transparency are real production advantages that shouldn't be discarded without confirming the accuracy gain is actually worth the added complexity and latency.

**Scenario-based**

- Q: Production monitoring shows a rising rate of emails being misclassified because they mention a negation term once, briefly, but are actually genuinely relevant. What's your fix path?
  A: This is the known structural weakness of binary flag matching. The fix: replace the binary negation flag with a BM25-scored negation signal, so a single incidental mention scores much lower than an email where the negation topic dominates the text — while keeping the exact same downstream branching logic so the output categories stay consistent.

**Follow-up questions to expect:**

- "How would you measure whether the BM25-scored upgrade is actually worth deploying?"
- "What happens to the rule engine's behavior as new product names or slang terms emerge that aren't in the keyword list yet?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Rule-based systems as a special case of statistical retrieval:** many hand-built rule systems can be understood as a simplified, manually-tuned approximation of a more general statistical method — recognizing this connection helps predict exactly where the simplified version will fail, before it fails in production.
- **The cost of manual curation vs automatic recalculation:** any system requiring humans to manually maintain term lists or weights (like this rule engine's keyword groups) will drift out of date as the underlying data changes, whereas statistically-driven methods like BM25 recompute their weights automatically whenever the corpus is reprocessed.
- **Combining scoring systems with domain logic:** a common, practical pattern in production classifiers is exactly what this topic's proposed upgrade demonstrates — use a statistical scoring method (like BM25) to get better signal quality, but keep the surrounding domain-specific decision logic (branching, thresholds, category routing) that encodes real business rules the raw statistical method doesn't know about.

### 10. Quick Revision Summary (for last-minute interview prep)

> A hand-rolled keyword classifier is a real, working example of sparse retrieval taken to its simplest extreme: check word presence, and stop. It shares BM25's starting point (does this term appear?) but skips frequency weighting (repetition doesn't matter once a match is found), rarity weighting (every keyword is treated as equally important), and length normalization (less relevant here anyway, since the emails are short and similarly sized). What the rule engine has that plain BM25 doesn't: genuine negation-based domain logic that routes to multiple output categories by combining two independent signals — something BM25's purely additive scoring has no native mechanism for. The natural upgrade path, if false positives from single incidental mentions become a real problem, is to replace the binary flags with BM25-based scores while keeping the exact same branching structure — gaining frequency and rarity awareness without discarding the domain logic that makes the system effective.


### Module 1: The Original Rule-Based Classifier

Rebuild the exact binary keyword-matching logic, applied to a few test emails.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: The original rule-based classifier
# ------------------------------------------------------------------

FD_KEYWORD_GROUPS = [
    ["fd", "fixed deposit", "maturity", "machurity", "byaj"],
    ["premature", "withdrawal", "nikalna", "naya fd"],
    ["nominee", "interest rate", "tenure"],
]

NEGATION_PHRASES = [
    "loan", "emi", "insurance", "card", "login", "otp",
    "app", "kyc", "branch", "statement",
]


def classify_by_keyword_rules(subject: str, content: str) -> str:
    """The ORIGINAL rule-based classifier: pure binary presence check,
    no frequency weighting, no rarity weighting."""
    text = (subject + " " + content).lower()
    has_fd_term = any(kw in text for group in FD_KEYWORD_GROUPS for kw in group)
    has_negation = any(neg in text for neg in NEGATION_PHRASES)

    if has_fd_term and not has_negation:
        return "FD"
    elif has_negation and not has_fd_term:
        return "Non-FD"
    elif has_fd_term and has_negation:
        return "Multiple Category"
    else:
        return "ambiguous"


TEST_EMAILS = [
    ("FD maturity query", "when will my fd mature and what is the interest rate"),
    ("Loan complaint", "my personal loan emi has bounced twice this month"),
    ("Mixed case", "I have applied for a personal loan against my FD please process fast"),
    ("Incidental mention", "by the way I also have a loan with you but that is fine, my real question is about my fd maturity date"),
    ("Login issue", "app login is not working otp is not coming"),
]

print("=" * 70)
print("MODULE 1: ORIGINAL RULE-BASED CLASSIFIER")
print("=" * 70)
for label, email in TEST_EMAILS:
    result = classify_by_keyword_rules("", email)
    print(f"  [{label:<20}] -> {result:<18} | \"{email[:55]}...\"")

print("\nWatch 'Incidental mention': the word 'loan' appears only once,")
print("briefly, but the classifier treats it EXACTLY as strongly as an")
print("email entirely about a loan -- it gets routed to 'Multiple Category'")
print("even though the email is really about FD maturity. This is the")
print("known structural weakness this topic is about.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: ORIGINAL RULE-BASED CLASSIFIER
  [FD maturity query   ] -> FD                 | "when will my fd mature and what is the interest rate..."
  [Loan complaint      ] -> Non-FD             | "my personal loan emi has bounced twice this month..."
  [Mixed case          ] -> Multiple Category  | "I have applied for a personal loan against my FD please..."
  [Incidental mention  ] -> Multiple Category  | "by the way I also have a loan with you but that is fine..."
  [Login issue         ] -> Non-FD             | "app login is not working otp is not coming..."

Watch 'Incidental mention': the word 'loan' appears only once,
briefly, but the classifier treats it EXACTLY as strongly as an
email entirely about a loan -- it gets routed to 'Multiple Category'
even though the email is really about FD maturity. This is the
known structural weakness this topic is about.

Module 1 complete. Run Module 2 next.


### Module 2: BM25-Scored Variant

Same decision structure, but the binary flags are replaced with BM25 relevance scores compared against a threshold. IMPORTANT: BM25 scores documents against a query, so here the EMAILS are the documents being scored, and the keyword groups are the queries — this is the direction that lets BM25's real term-frequency and rarity math actually apply per email.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: BM25-scored variant of the same classifier
#
# CORRECT DIRECTION: BM25 saturates and weights term frequency on the
# DOCUMENT side, not the query side. To score "how strongly does this
# email indicate FD / negation", the EMAILS must be the documents in
# the BM25 index, and the keyword-group text is the query used to
# score each email. This also gives a realistic multi-document corpus
# for IDF, avoiding the single-document negative-IDF edge case from
# Topic 1.
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()

fd_query_tokens = tokenize(" ".join(kw for group in FD_KEYWORD_GROUPS for kw in group))
negation_query_tokens = tokenize(" ".join(NEGATION_PHRASES))

# the EMAILS are the documents -- this is the correct BM25 direction
email_texts = [email for _, email in TEST_EMAILS]
tokenized_emails = [tokenize(e) for e in email_texts]
email_bm25 = BM25Okapi(tokenized_emails)

fd_scores = email_bm25.get_scores(fd_query_tokens)
negation_scores = email_bm25.get_scores(negation_query_tokens)

THRESHOLD = 0.1

print("=" * 70)
print("MODULE 2: BM25-SCORED VARIANT (emails scored as documents)")
print("=" * 70)
print(f"{'Email':<20} | {'Original':<18} | {'BM25 result':<18} | {'FD score':>8} | {'Neg score':>9}")
print("-" * 85)
for i, (label, email) in enumerate(TEST_EMAILS):
    original = classify_by_keyword_rules("", email)
    fd_score = fd_scores[i]
    neg_score = negation_scores[i]
    has_fd = fd_score > THRESHOLD
    has_neg = neg_score > THRESHOLD
    if has_fd and not has_neg:
        bm25_label = "FD"
    elif has_neg and not has_fd:
        bm25_label = "Non-FD"
    elif has_fd and has_neg:
        bm25_label = "Multiple Category"
    else:
        bm25_label = "ambiguous"
    print(f"{label:<20} | {original:<18} | {bm25_label:<18} | {fd_score:>8.3f} | {neg_score:>9.3f}")

print("\nWith a realistic multi-document corpus (the 5 test emails), BM25")
print("now produces meaningful positive scores instead of the degenerate")
print("near-zero/negative scores a single-document corpus would give")
print("(that single-document case IS the negative-IDF edge case from")
print("Topic 1 -- a real trap when building a BM25 index this way).")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: BM25-SCORED VARIANT (emails scored as documents)
Email                | Original           | BM25 result        | FD score | Neg score
-------------------------------------------------------------------------------------
FD maturity query    | FD                 | FD                 |    2.833 |     0.000
Loan complaint       | Non-FD             | Non-FD             |    0.000 |     1.530
Mixed case           | Multiple Category  | Multiple Category  |    0.440 |     0.220
Incidental mention   | Multiple Category  | Multiple Category  |    1.143 |     0.163
Login issue          | Non-FD             | Non-FD             |    0.000 |     3.826

With a realistic multi-document corpus (the 5 test emails), BM25
now produces meaningful positive scores instead of the degenerate
near-zero/negative scores a single-document corpus would give
(that single-document case IS the negative-IDF edge case from
Topic 1 -- a real trap when building a BM25 index this way).

Module 2 complete. Ru

### Module 3: Frequency Sensitivity — What the Binary Version Cannot See

Directly proves the theory's core claim, with BM25 used in the CORRECT direction: the repetition test emails are the documents, so document-side term-frequency saturation genuinely applies.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Frequency sensitivity comparison
#
# CORRECT DIRECTION (same fix as Module 2): the repetition-test
# emails are the DOCUMENTS in the BM25 index, and "loan" is the
# query. This way, BM25's real per-document term-frequency
# saturation applies, instead of query-side repetition (which
# just sums linearly and does not saturate at all).
# ------------------------------------------------------------------

REPETITION_TEST_EMAILS = [
    ("loan mentioned once",    "i have a loan with you as well please note"),
    ("loan mentioned 3 times", "loan loan loan payment is due please help with my loan"),
    ("loan mentioned 6 times", "loan loan loan loan loan loan is overdue please help"),
]

repetition_texts = [email for _, email in REPETITION_TEST_EMAILS]
tokenized_repetition = [tokenize(e) for e in repetition_texts]
repetition_bm25 = BM25Okapi(tokenized_repetition)

loan_scores = repetition_bm25.get_scores(tokenize("loan"))

print("=" * 70)
print("MODULE 3: FREQUENCY SENSITIVITY -- binary flag vs BM25 score")
print("=" * 70)
print(f"{'Test case':<25} | {'Original has_negation':<22} | {'BM25 score (doc-side TF)':>24}")
print("-" * 78)

for i, (label, email) in enumerate(REPETITION_TEST_EMAILS):
    text = email.lower()
    original_flag = any(neg in text for neg in NEGATION_PHRASES)
    print(f"{label:<25} | {str(original_flag):<22} | {loan_scores[i]:>24.3f}")

print("\nThe original binary flag is IDENTICAL (True) across all three cases")
print("-- it cannot tell 'mentioned once' apart from 'mentioned six times'.")
print("The BM25 score now genuinely increases with real document-side")
print("repetition, but with DIMINISHING returns per additional repeat --")
print(f"going from 1 to 3 mentions changed the score by {loan_scores[1]-loan_scores[0]:.3f},")
print(f"but going from 3 to 6 mentions only changed it by {loan_scores[2]-loan_scores[1]:.3f}")
print("-- the same k1 saturation curve proven with real numbers back in")
print("Topic 1, Module 5. This only shows up correctly because the")
print("repeated word lives in the DOCUMENT being scored, not the query.")
print("\nModule 3 complete. All Topic 5 modules done.")

print()
print("=" * 70)
print("CHAPTER 7 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  The rule-based classifier IS a form of sparse retrieval -- the
  simplest possible one: binary term presence, nothing else.

  It is missing vs BM25: frequency weighting, rarity weighting,
  length normalization (less relevant for short, similar-length emails).

  It HAS something BM25 lacks: negation-based domain logic that routes
  to multiple output categories by combining two independent signals.

  IMPORTANT implementation detail: BM25 saturates DOCUMENT-side term
  frequency, not query-side repetition -- get the direction right,
  or the saturation behavior will not show up at all.

  Natural upgrade path if false positives from single incidental
  mentions become a real problem: replace binary flags with BM25
  scores above a threshold, keeping the exact same branching logic.
""")


MODULE 3: FREQUENCY SENSITIVITY -- binary flag vs BM25 score
Test case                 | Original has_negation  | BM25 score (doc-side TF)
------------------------------------------------------------------------------
loan mentioned once       | True                   |                    0.003
loan mentioned 3 times    | True                   |                    0.005
loan mentioned 6 times    | True                   |                    0.006

The original binary flag is IDENTICAL (True) across all three cases
-- it cannot tell 'mentioned once' apart from 'mentioned six times'.
The BM25 score now genuinely increases with real document-side
repetition, but with DIMINISHING returns per additional repeat --
going from 1 to 3 mentions changed the score by 0.002,
but going from 3 to 6 mentions only changed it by 0.001
-- the same k1 saturation curve proven with real numbers back in
Topic 1, Module 5. This only shows up correctly because the
repeated word lives in the DOCUMENT being sco